In [ ]:
import safetensors

In [4]:
from transformers import AutoModelForCausalLM
import torch
from collections import OrderedDict
import safetensors # Use the correct library to load the file

# --- CONFIGURATION ---
model_architecture_path = "/home/jorge/tokenPred/babylm_10m/train_files/gpt-bert/model_checkpoints/gptbert_arc"
weights_path = "/home/jorge/tokenPred/babylm_10m/train_files/gpt-bert/model_checkpoints/gptbert_arc/model.safetensors"

# --- SCRIPT ---

# 1. Build the model architecture (this part is correct)
print("--- Building final model structure ---")
model = AutoModelForCausalLM.from_pretrained(
    model_architecture_path,
    trust_remote_code=True
)
print("✅ Model architecture built successfully.\n")

# 2. Load the weights using the 'safetensors' library
print(f"--- Loading weights from {weights_path} ---")
state_dict = safetensors.torch.load_file(weights_path) # <-- THE FIX IS HERE
print("✅ Weights loaded successfully using safetensors.\n")

# 3. Create a new dictionary and fix the prefix
print("--- Fixing weight key prefixes ---")
new_state_dict = OrderedDict()
for k, v in state_dict.items():
    if k.startswith("transformer."):
        new_key = "model." + k[len("transformer."):]
        new_state_dict[new_key] = v
    else:
        new_state_dict[k] = v
print("✅ Prefixes corrected.\n")

# 4. Load the corrected weights into the model
print("--- Applying final weights to model ---")
model.load_state_dict(new_state_dict, strict=False)
model.eval()
print("🚀🚀🚀 Success! The model is fully loaded with the correct weights.")

--- Building final model structure ---


Some weights of GPTBERTForCausalLM were not initialized from the model checkpoint at /home/jorge/tokenPred/babylm_10m/train_files/gpt-bert/model_checkpoints/gptbert_arc and are newly initialized: ['lm_head.nonlinearity.1.bias', 'lm_head.nonlinearity.1.weight', 'lm_head.nonlinearity.5.bias', 'lm_head.nonlinearity.5.weight', 'model.attention_layers.0.in_proj_qk.bias', 'model.attention_layers.0.in_proj_qk.weight', 'model.attention_layers.0.in_proj_vg.bias', 'model.attention_layers.0.in_proj_vg.weight', 'model.attention_layers.0.out_proj.bias', 'model.attention_layers.0.out_proj.weight', 'model.attention_layers.1.in_proj_qk.bias', 'model.attention_layers.1.in_proj_qk.weight', 'model.attention_layers.1.in_proj_vg.bias', 'model.attention_layers.1.in_proj_vg.weight', 'model.attention_layers.1.out_proj.bias', 'model.attention_layers.1.out_proj.weight', 'model.attention_layers.10.in_proj_qk.bias', 'model.attention_layers.10.in_proj_qk.weight', 'model.attention_layers.10.in_proj_vg.bias', 'model

✅ Model architecture built successfully.

--- Loading weights from /home/jorge/tokenPred/babylm_10m/train_files/gpt-bert/model_checkpoints/gptbert_arc/model.safetensors ---
✅ Weights loaded successfully using safetensors.

--- Fixing weight key prefixes ---
✅ Prefixes corrected.

--- Applying final weights to model ---
🚀🚀🚀 Success! The model is fully loaded with the correct weights.


In [5]:
from tokenizers import Tokenizer
from transformers import PreTrainedTokenizerFast
# Load the tokenizer from a file
tokenizer = Tokenizer.from_file("/home/jorge/tokenPred/babylm_10m/train_files/gpt-bert/tokenizers/tokenizer_10M.json")
tokenizer = PreTrainedTokenizerFast(tokenizer_object=tokenizer)

In [6]:
model.save_pretrained("/home/jorge/tokenPred/babylm_10m/train_files/gpt-bert/model_checkpoints/gptbert_upload")
tokenizer.save_pretrained("/home/jorge/tokenPred/babylm_10m/train_files/gpt-bert/model_checkpoints/gptbert_upload")

('/home/jorge/tokenPred/babylm_10m/train_files/gpt-bert/model_checkpoints/gptbert_upload/tokenizer_config.json',
 '/home/jorge/tokenPred/babylm_10m/train_files/gpt-bert/model_checkpoints/gptbert_upload/special_tokens_map.json',
 '/home/jorge/tokenPred/babylm_10m/train_files/gpt-bert/model_checkpoints/gptbert_upload/tokenizer.json')

In [8]:
from transformers import AutoModelForCausalLM
m = AutoModelForCausalLM.from_pretrained(
    "/home/jorge/tokenPred/babylm_10m/train_files/gpt-bert/model_checkpoints/gptbert_upload",
    trust_remote_code=True
)
print(type(m))  # should be your GPTBERT class


<class 'transformers_modules.gptbert_upload.modeling_gpt_bert.GPTBERTForCausalLM'>


In [ ]:
from huggingface_hub import login
from transformers import PreTrainedTokenizerFast
from tokenizers import Tokenizer
from transformers import AutoTokenizer, AutoModelForCausalLM

login()  # Replace with your Hugging Face token

# Save the model and tokenizer to the Hugging Face Hub
model.push_to_hub("jorge-babylm-10m-gptbert", use_auth_token=True)
tokenizer.push_to_hub("jorge-babylm-10m-gptbert", use_auth_token=True)

/home/jorge/miniconda3/envs/babylm/lib/python3.11/site-packages/transformers/utils/hub.py:920: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


model.safetensors:   0%|          | 0.00/528M [00:00<?, ?B/s]

/home/jorge/miniconda3/envs/babylm/lib/python3.11/site-packages/transformers/utils/hub.py:920: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/Jorgeis1/jorge-babylm-10m-gptbert/commit/61c6eb93e096f5c90e479716dd6f9a7d4f447ce0', commit_message='Upload tokenizer', commit_description='', oid='61c6eb93e096f5c90e479716dd6f9a7d4f447ce0', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Jorgeis1/jorge-babylm-10m-gptbert', endpoint='https://huggingface.co', repo_type='model', repo_id='Jorgeis1/jorge-babylm-10m-gptbert'), pr_revision=None, pr_num=None)

In [ ]:
from huggingface_hub import login, HfApi, create_repo
import os, json, shutil

login()  # don't hardcode tokens in notebooks

repo_id = "jorgeis1/jorge-babylm-10m-gptbert"   # be explicit with owner/name
out_dir = "/home/jorge/tokenPred/babylm_10m/train_files/gpt-bert/model_checkpoints/gptbert_upload"

# 1) Save weights & config
model.save_pretrained(out_dir, safe_serialization=True)
tokenizer.save_pretrained(out_dir)

# 2) Make sure your code files are present
src = "/home/jorge/tokenPred/babylm_10m/train_files/gpt-bert/model_checkpoints/gptbert_arc"
for fn in ["modeling_gpt_bert.py", "configuration_gpt_bert.py"]:
    shutil.copy2(os.path.join(src, fn), os.path.join(out_dir, fn))
open(os.path.join(out_dir, "__init__.py"), "a").close()

# 3) Ensure config has model_type + auto_map
cfg_path = os.path.join(out_dir, "config.json")
cfg = json.load(open(cfg_path))
cfg.setdefault("model_type", "gpt-bert")
cfg["auto_map"] = {
    "AutoConfig": "configuration_gpt_bert.ModelConfig",
    "AutoModel": "modeling_gpt_bert.GPTBERT",
    "AutoModelForCausalLM": "modeling_gpt_bert.GPTBERTForCausalLM",
    "AutoModelForMaskedLM": "modeling_gpt_bert.GPTBERTForMaskedLM",
}
json.dump(cfg, open(cfg_path, "w"), indent=2)

# 4) Create repo and upload the whole folder
api = HfApi()
create_repo(repo_id, exist_ok=True)  # default private=False; set as you like
api.upload_folder(
    folder_path=out_dir,
    repo_id=repo_id,
    commit_message="Initial: weights + tokenizer + custom modeling code"
)

# 5) Sanity check (remote)
from transformers import AutoModelForCausalLM
m = AutoModelForCausalLM.from_pretrained(repo_id, trust_remote_code=True)
print(type(m))


No files have been modified since last commit. Skipping to prevent empty commit.


<class 'transformers_modules.jorgeis1.jorge-babylm-10m-gptbert.e97eff8b2c8f278e728ef8b465941c1c96866aaf.modeling_gpt_bert.GPTBERTForCausalLM'>


In [ ]:
# --- settings ---
REPO_ID = "jorgeis1/jorge-babylm-10m-gptbert"
OUT = "/home/jorge/tokenPred/babylm_10m/train_files/gpt-bert/model_checkpoints/gptbert_upload"

from transformers import PreTrainedTokenizerFast
from huggingface_hub import login, HfApi, create_repo
import json, os

login()  # will use cached token; if not logged in, it will prompt
#login('')  # uses cached token

# 1) Load tokenizer from the folder you’re pushing (it already has tokenizer.json)
tok = PreTrainedTokenizerFast.from_pretrained(OUT)

# 2) Ensure the specials you want are set/mapped
tok.bos_token  = "<s>"
tok.eos_token  = "</s>"
tok.unk_token  = "<unk>"
tok.pad_token  = "<pad>"
tok.mask_token = "<mask>"
tok.sep_token  = "</s>"
tok.cls_token  = "<s>"
tok.model_max_length = 512

# 3) Save tokenizer + an explicit tokenizer_config.json with tokenizer_class
tok.save_pretrained(OUT)
cfg = {
    "tokenizer_class": "PreTrainedTokenizerFast",
    "model_max_length": tok.model_max_length
}
json.dump(cfg, open(os.path.join(OUT, "tokenizer_config.json"), "w"), indent=2)

# 4) Push folder again
api = HfApi(); create_repo(repo_id=REPO_ID, exist_ok=True)
api.upload_folder(folder_path=OUT, repo_id=REPO_ID, repo_type="model",
                  commit_message="Fix tokenizer_config: add tokenizer_class; keep special tokens")

# 5) Sanity check (remote)
from transformers import AutoTokenizer
rtok = AutoTokenizer.from_pretrained(REPO_ID, trust_remote_code=True)
print("OK:", rtok.bos_token, rtok.eos_token, rtok.pad_token, rtok.mask_token)


No files have been modified since last commit. Skipping to prevent empty commit.


OK: <s> </s> <pad> <mask>
